In [ ]:
import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("Running without a GPU.")

except ImportError:
    print("PyTorch is not installed yet.")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
%pip install -q FlagEmbedding faiss-cpu

In [ ]:
import torch
import faiss
import FlagEmbedding
import transformers

print("PyTorch:", torch.__version__)
print("FAISS:", faiss.__version__)
print("Transformers:", transformers.__version__)
print("FlagEmbedding imported successfully!")
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
FAISS: 1.15.1
Transformers: 5.20.2
FlagEmbedding imported successfully!
GPU available: True


In [ ]:
import torch
from FlagEmbedding import BGEM3FlagModel

MODEL_NAME = "BAAI/bge-m3"

print("Loading BGE-M3...")

bge_model = BGEM3FlagModel(
    MODEL_NAME,
    use_fp16=True,
    devices=["cuda:0"]
)

print("\nBGE-M3 loaded successfully!")
print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))

Loading BGE-M3...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

... (2 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


BGE-M3 loaded successfully!
Model: BAAI/bge-m3
GPU: Tesla T4


In [ ]:
import numpy as np
import unicodedata

# Sample sentences for a technical smoke test.
# These are not real TeluguPeople news articles.

sentences = [
    "What are the latest developments in technology?",
    "సాంకేతిక రంగంలో తాజా పరిణామాలు ఏమిటి?",
    "What are the తాజా developments in technology?"
]

# Apply the same Unicode normalization rule to every input.
sentences = [
    unicodedata.normalize("NFC", text).strip()
    for text in sentences
]

print("Generating embeddings...")

output = bge_model.encode(
    sentences,
    batch_size=3,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

embeddings = np.asarray(output["dense_vecs"], dtype=np.float32)

print("\nEmbedding verification")
print("----------------------")
print("Number of sentences:", len(sentences))
print("Embedding shape:", embeddings.shape)
print("Expected shape:", (3, 1024))
print("All values finite:", np.isfinite(embeddings).all())
print("All vectors nonzero:", np.all(np.linalg.norm(embeddings, axis=1) > 0))

assert embeddings.shape == (3, 1024), (
    f"Unexpected embedding shape: {embeddings.shape}"
)
assert np.isfinite(embeddings).all(), "Non-finite embedding values detected."
assert np.all(np.linalg.norm(embeddings, axis=1) > 0), "Zero vector detected."

print("\nAll embedding checks passed!")

Generating embeddings...


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 53.14it/s]


Embedding verification
----------------------
Number of sentences: 3
Embedding shape: (3, 1024)
Expected shape: (3, 1024)
All values finite: True
All vectors nonzero: True

All embedding checks passed!


In [ ]:
import unicodedata

demo_chunks = [
    {
        "chunk_id": "demo-en-001",
        "text": (
            "The Indian government announced new initiatives "
            "to support technology development and digital services."
        )
    },
    {
        "chunk_id": "demo-te-001",
        "text": (
            "భారత ప్రభుత్వం సాంకేతిక అభివృద్ధి మరియు "
            "డిజిటల్ సేవలకు మద్దతుగా కొత్త కార్యక్రమాలను ప్రకటించింది."
        )
    },
    {
        "chunk_id": "demo-en-002",
        "text": (
            "A cricket match between India and Australia "
            "attracted a large audience."
        )
    },
    {
        "chunk_id": "demo-mix-001",
        "text": (
            "Indiaలో digital services మరియు technology development "
            "గురించి ప్రభుత్వం కొత్త initiatives ప్రకటించింది."
        )
    },
    {
        "chunk_id": "demo-en-003",
        "text": (
            "Heavy rainfall affected transportation and daily life "
            "in several regions."
        )
    }
]

# Apply the same Unicode normalization to every chunk.
for chunk in demo_chunks:
    chunk["text"] = unicodedata.normalize(
        "NFC", chunk["text"]
    ).strip()

# Check that chunk IDs are unique.
chunk_ids = [chunk["chunk_id"] for chunk in demo_chunks]

assert len(chunk_ids) == len(set(chunk_ids))
assert all(chunk["text"] for chunk in demo_chunks)

print("Number of demonstration chunks:", len(demo_chunks))
print("Unique chunk IDs:", len(set(chunk_ids)))

for chunk in demo_chunks:
    print(chunk["chunk_id"], ":", chunk["text"][:70])

Number of demonstration chunks: 5
Unique chunk IDs: 5
demo-en-001 : The Indian government announced new initiatives to support technology 
demo-te-001 : భారత ప్రభుత్వం సాంకేతిక అభివృద్ధి మరియు డిజిటల్ సేవలకు మద్దతుగా కొత్త 
demo-en-002 : A cricket match between India and Australia attracted a large audience
demo-mix-001 : Indiaలో digital services మరియు technology development గురించి ప్రభుత్వ
demo-en-003 : Heavy rainfall affected transportation and daily life in several regio


In [ ]:
import numpy as np

chunk_texts = [
    chunk["text"] for chunk in demo_chunks
]

print("Embedding demonstration chunks...")

document_output = bge_model.encode(
    chunk_texts,
    batch_size=4,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

document_embeddings = np.asarray(
    document_output["dense_vecs"],
    dtype=np.float32
)

assert document_embeddings.shape == (len(demo_chunks), 1024)
assert np.isfinite(document_embeddings).all()

print("Embedding shape:", document_embeddings.shape)
print("Embedding generation successful!")

Embedding demonstration chunks...


Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00, 14.55it/s]

Embedding shape: (5, 1024)
Embedding generation successful!


In [ ]:
import faiss

# Make a separate copy before normalizing.
document_vectors = document_embeddings.copy()

# Normalize vectors so inner product equals cosine similarity.
faiss.normalize_L2(document_vectors)

embedding_dimension = document_vectors.shape[1]

# Exact similarity-search index.
index = faiss.IndexFlatIP(embedding_dimension)

# Add one vector per chunk.
index.add(document_vectors)

print("FAISS index created successfully!")
print("Embedding dimension:", index.d)
print("Vectors in index:", index.ntotal)
print("Chunks in dataset:", len(demo_chunks))

assert index.ntotal == len(demo_chunks)

print("FAISS vector-count verification passed!")

FAISS index created successfully!
Embedding dimension: 1024
Vectors in index: 5
Chunks in dataset: 5
FAISS vector-count verification passed!


In [ ]:
import time
import unicodedata
import numpy as np
import faiss

def retrieve_chunks(question, top_k=3):
    question = unicodedata.normalize(
        "NFC", question
    ).strip()

    if not question:
        raise ValueError("Question cannot be empty.")

    if top_k < 1:
        raise ValueError("top_k must be at least 1.")

    # Measure query embedding and FAISS search together.
    start_time = time.perf_counter()

    query_output = bge_model.encode(
        [question],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    query_vector = np.asarray(
        query_output["dense_vecs"],
        dtype=np.float32
    )

    if query_vector.shape != (1, embedding_dimension):
        raise ValueError(
            f"Unexpected query embedding shape: {query_vector.shape}"
        )

    if not np.isfinite(query_vector).all():
        raise ValueError("Query embedding contains non-finite values.")

    faiss.normalize_L2(query_vector)

    scores, positions = index.search(
        query_vector,
        min(top_k, index.ntotal)
    )

    elapsed_ms = (time.perf_counter() - start_time) * 1000

    results = []

    for score, position in zip(
        scores[0].tolist(),
        positions[0].tolist()
    ):
        if position < 0:
            continue

        results.append({
            "chunk_id": demo_chunks[position]["chunk_id"],
            "score": float(score),
            "text": demo_chunks[position]["text"]
        })

    return {
        "question": question,
        "retrieved_chunk_ids": [
            item["chunk_id"] for item in results
        ],
        "retrieved_scores": [
            item["score"] for item in results
        ],
        "retrieval_latency_ms": round(elapsed_ms, 3),
        "results": results
    }


test_result = retrieve_chunks(
    "What did the Indian government announce about technology?",
    top_k=3
)

print("Question:", test_result["question"])
print("Retrieval latency:", test_result["retrieval_latency_ms"], "ms")

print("\nRanked results:")

for rank, result in enumerate(test_result["results"], start=1):
    print(f"\nRank {rank}")
    print("Chunk ID:", result["chunk_id"])
    print("Similarity score:", round(result["score"], 4))
    print("Text:", result["text"])

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 61.75it/s]

Question: What did the Indian government announce about technology?
Retrieval latency: 51.136 ms

Ranked results:

Rank 1
Chunk ID: demo-en-001
Similarity score: 0.7715
Text: The Indian government announced new initiatives to support technology development and digital services.

Rank 2
Chunk ID: demo-mix-001
Similarity score: 0.7313
Text: Indiaలో digital services మరియు technology development గురించి ప్రభుత్వం కొత్త initiatives ప్రకటించింది.

Rank 3
Chunk ID: demo-te-001
Similarity score: 0.7079
Text: భారత ప్రభుత్వం సాంకేతిక అభివృద్ధి మరియు డిజిటల్ సేవలకు మద్దతుగా కొత్త కార్యక్రమాలను ప్రకటించింది.


In [ ]:
import json
import hashlib
import faiss
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

# Create an output directory.
output_dir = Path("/content/bge_m3_artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the FAISS index.
faiss.write_index(
    index,
    str(output_dir / "bge_m3.faiss")
)

# Save the chunks in their existing order.
chunks_path = output_dir / "chunks.jsonl"

with chunks_path.open("w", encoding="utf-8") as f:
    for chunk in demo_chunks:
        f.write(
            json.dumps(chunk, ensure_ascii=False) + "\n"
        )

# Calculate a hash of the exact saved chunk file.
chunk_hash = hashlib.sha256(
    chunks_path.read_bytes()
).hexdigest()

# Create the initial run manifest.
manifest = {
    "model_name": "BAAI/bge-m3",
    "model_revision": "not pinned",
    "embedding_dimension": int(index.d),
    "embedding_representation": "dense",
    "similarity_metric": "cosine",
    "faiss_index_type": "IndexFlatIP",
    "unicode_normalization": "NFC",
    "query_prefix": None,
    "document_prefix": None,
    "max_length_used": 512,
    "top_k": 3,
    "chunk_file_sha256": chunk_hash,
    "indexed_chunk_count": len(demo_chunks),
    "faiss_vector_count": int(index.ntotal),
    "generator_model": None,
    "generation_settings": None,
    "dataset_type": "fabricated demonstration data",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}

manifest_path = output_dir / "manifest.json"

manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# Verify the saved index.
saved_index = faiss.read_index(
    str(output_dir / "bge_m3.faiss")
)

assert saved_index.ntotal == len(demo_chunks)

print("Artifacts saved successfully!")
print("Index vectors:", saved_index.ntotal)
print("Chunk file SHA-256:", chunk_hash)
print("Output directory:", output_dir)
print("\nFiles:")

for path in sorted(output_dir.iterdir()):
    print("-", path.name)

Artifacts saved successfully!
Index vectors: 5
Chunk file SHA-256: ab16eedc5cd0e4274848705332e9e3b4b01b01469ac10674b9c24e65e6a1b00e
Output directory: /content/bge_m3_artifacts

Files:
- bge_m3.faiss
- chunks.jsonl
- manifest.json


In [ ]:
import json
from pathlib import Path

evaluation_examples = [
    {
        "question_id": "demo-q-001",
        "question": "What did the Indian government announce about technology?",
        "language": "en",
        "reference_answer": (
            "The government announced initiatives supporting "
            "technology development and digital services."
        ),
        "relevant_chunk_ids": [
            "demo-en-001",
            "demo-mix-001",
            "demo-te-001"
        ]
    },
    {
        "question_id": "demo-q-002",
        "question": "సాంకేతిక అభివృద్ధికి ప్రభుత్వం ఏమి ప్రకటించింది?",
        "language": "te",
        "reference_answer": (
            "సాంకేతిక అభివృద్ధి మరియు డిజిటల్ సేవలకు "
            "మద్దతుగా కొత్త కార్యక్రమాలను ప్రకటించింది."
        ),
        "relevant_chunk_ids": [
            "demo-te-001",
            "demo-en-001",
            "demo-mix-001"
        ]
    },
    {
        "question_id": "demo-q-003",
        "question": "What initiatives ప్రభుత్వం technology కోసం ప్రకటించింది?",
        "language": "mixed",
        "reference_answer": (
            "The government announced initiatives supporting "
            "technology development and digital services."
        ),
        "relevant_chunk_ids": [
            "demo-mix-001",
            "demo-en-001",
            "demo-te-001"
        ]
    }
]

evaluation_path = Path(
    "/content/bge_m3_artifacts/evaluation_demo.jsonl"
)

with evaluation_path.open("w", encoding="utf-8") as f:
    for example in evaluation_examples:
        f.write(
            json.dumps(example, ensure_ascii=False) + "\n"
        )

print("Evaluation format created!")
print("Example questions:", len(evaluation_examples))
print("File:", evaluation_path)

for example in evaluation_examples:
    print(
        example["question_id"],
        "| Language:", example["language"]
    )

Evaluation format created!
Example questions: 3
File: /content/bge_m3_artifacts/evaluation_demo.jsonl
demo-q-001 | Language: en
demo-q-002 | Language: te
demo-q-003 | Language: mixed


In [ ]:
import json
from pathlib import Path

chunks_path = Path(
    "/content/bge_m3_artifacts/chunks.jsonl"
)

evaluation_path = Path(
    "/content/bge_m3_artifacts/evaluation_demo.jsonl"
)

with chunks_path.open(encoding="utf-8") as f:
    available_chunk_ids = {
        json.loads(line)["chunk_id"]
        for line in f if line.strip()
    }

with evaluation_path.open(encoding="utf-8") as f:
    questions = [
        json.loads(line)
        for line in f if line.strip()
    ]

question_ids = [q["question_id"] for q in questions]

assert len(question_ids) == len(set(question_ids)), (
    "Duplicate question IDs found."
)

for question in questions:
    missing_ids = (
        set(question["relevant_chunk_ids"])
        - available_chunk_ids
    )

    assert not missing_ids, (
        f"{question['question_id']} references "
        f"unknown chunks: {missing_ids}"
    )

print("Evaluation dataset structure verified.")
print("Unique questions:", len(questions))
print("All referenced chunk IDs exist.")

Evaluation dataset structure verified.
Unique questions: 3
All referenced chunk IDs exist.


In [ ]:
import json
import time
import unicodedata
from pathlib import Path
from collections import defaultdict

import faiss
import numpy as np

# --------------------------------------------------
# 1. Load saved artifacts
# --------------------------------------------------

ARTIFACT_DIR = Path("/content/bge_m3_artifacts")

with open(ARTIFACT_DIR / "chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

with open(ARTIFACT_DIR / "evaluation_demo.jsonl", "r", encoding="utf-8") as f:
    questions = [json.loads(line) for line in f if line.strip()]

index = faiss.read_index(str(ARTIFACT_DIR / "bge_m3.faiss"))

# Adjust these keys only if your JSONL uses different names.
chunk_ids = [c["chunk_id"] for c in chunks]
chunk_texts = [c["text"] for c in chunks]

assert len(chunks) == index.ntotal, (
    f"Mismatch: {len(chunks)} chunks but {index.ntotal} index vectors."
)

chunk_id_to_position = {
    chunk_id: i for i, chunk_id in enumerate(chunk_ids)
}

print("Chunks loaded:", len(chunks))
print("Questions loaded:", len(questions))
print("FAISS vectors:", index.ntotal)


# --------------------------------------------------
# 2. Prepare query embeddings
# --------------------------------------------------

def normalize_text(text):
    return unicodedata.normalize("NFC", text).strip()


def embed_query(question):
    encoded = bge_model.encode(
        [normalize_text(question)],
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    vector = np.asarray(
        encoded["dense_vecs"], dtype=np.float32
    )

    faiss.normalize_L2(vector)
    return vector


# --------------------------------------------------
# 3. Retrieve chunks and calculate metrics
# --------------------------------------------------

TOP_K = min(3, index.ntotal)
raw_results = []
all_recall = []
all_rr = []
metrics_by_language = defaultdict(
    lambda: {"recall": [], "rr": []}
)

for item in questions:
    question_id = item["question_id"]
    question_text = item["question"]
    language = item.get("language", "unknown")

    relevant_ids = set(item["relevant_chunk_ids"])

    # Check that every relevance label refers to a real chunk.
    missing_ids = relevant_ids - set(chunk_id_to_position)
    assert not missing_ids, (
        f"{question_id} has missing relevant chunk IDs: {missing_ids}"
    )

    query_vector = embed_query(question_text)

    start = time.perf_counter()
    scores, positions = index.search(query_vector, TOP_K)
    latency_ms = (time.perf_counter() - start) * 1000

    retrieved = []

    for rank, (position, score) in enumerate(
        zip(positions[0], scores[0]), start=1
    ):
        if position < 0:
            continue

        chunk = chunks[int(position)]

        retrieved.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "score": float(score),
            "text": chunk["text"]
        })

    retrieved_ids = [r["chunk_id"] for r in retrieved]

    # Recall@k: proportion of relevant chunks retrieved.
    recall = (
        len(set(retrieved_ids) & relevant_ids) / len(relevant_ids)
        if relevant_ids else 0.0
    )

    # Reciprocal rank: inverse rank of the first relevant result.
    reciprocal_rank = 0.0

    for result in retrieved:
        if result["chunk_id"] in relevant_ids:
            reciprocal_rank = 1.0 / result["rank"]
            break

    all_recall.append(recall)
    all_rr.append(reciprocal_rank)

    metrics_by_language[language]["recall"].append(recall)
    metrics_by_language[language]["rr"].append(reciprocal_rank)

    raw_results.append({
        "question_id": question_id,
        "question": question_text,
        "language": language,
        "relevant_chunk_ids": sorted(relevant_ids),
        "retrieved": retrieved,
        f"recall_at_{TOP_K}": recall,
        "reciprocal_rank": reciprocal_rank,
        "retrieval_latency_ms": latency_ms
    })


# --------------------------------------------------
# 4. Calculate aggregate and language-level metrics
# --------------------------------------------------

overall_metrics = {
    f"recall_at_{TOP_K}": float(np.mean(all_recall)),
    "mrr": float(np.mean(all_rr)),
    "question_count": len(questions),
    "top_k": TOP_K,
    "model": "BAAI/bge-m3",
    "dataset": "fabricated_demo"
}

language_metrics = {}

for language, values in metrics_by_language.items():
    language_metrics[language] = {
        f"recall_at_{TOP_K}": float(np.mean(values["recall"])),
        "mrr": float(np.mean(values["rr"])),
        "question_count": len(values["rr"])
    }


# --------------------------------------------------
# 5. Save raw results and metrics
# --------------------------------------------------

results_path = ARTIFACT_DIR / "retrieval_results_demo.jsonl"
metrics_path = ARTIFACT_DIR / "evaluation_metrics_demo.json"

with open(results_path, "w", encoding="utf-8") as f:
    for result in raw_results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

report = {
    "overall": overall_metrics,
    "by_language": language_metrics,
    "warning": (
        "Demonstration results only. Questions and relevance labels "
        "are fabricated and must not be used as a real benchmark."
    )
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\n========== DEMO EVALUATION ==========")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("\nRaw results saved to:", results_path)
print("Metrics saved to:", metrics_path)

Chunks loaded: 5
Questions loaded: 3
FAISS vectors: 5

========== DEMO EVALUATION ==========
{
  "overall": {
    "recall_at_3": 1.0,
    "mrr": 1.0,
    "question_count": 3,
    "top_k": 3,
    "model": "BAAI/bge-m3",
    "dataset": "fabricated_demo"
  },
  "by_language": {
    "en": {
      "recall_at_3": 1.0,
      "mrr": 1.0,
      "question_count": 1
    },
    "te": {
      "recall_at_3": 1.0,
      "mrr": 1.0,
      "question_count": 1
    },
    "mixed": {
      "recall_at_3": 1.0,
      "mrr": 1.0,
      "question_count": 1
    }
  },
  "warning": "Demonstration results only. Questions and relevance labels are fabricated and must not be used as a real benchmark."
}

Raw results saved to: /content/bge_m3_artifacts/retrieval_results_demo.jsonl
Metrics saved to: /content/bge_m3_artifacts/evaluation_metrics_demo.json


In [ ]:
import json
import time
from pathlib import Path
import unicodedata
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss
import numpy as np

ARTIFACT_DIR = Path("/content/bge_m3_artifacts")
E5_MODEL_NAME = "intfloat/multilingual-e5-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {E5_MODEL_NAME} onto {device}...")

e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
e5_model = AutoModel.from_pretrained(E5_MODEL_NAME).to(device)
e5_model.eval()

def mean_pooling(model_output, attention_mask):
    """Perform average pooling with attention mask weighting."""
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def embed_e5(texts, is_query=False):
    """
    Multilingual-E5 strictly requires:
      - 'passage: <text>' for documents
      - 'query: <text>' for search queries
    """
    prefix = "query: " if is_query else "passage: "
    prefixed_texts = [prefix + unicodedata.normalize("NFC", t).strip() for t in texts]

    encoded = e5_tokenizer(
        prefixed_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = e5_model(**encoded)
        embeddings = mean_pooling(output, encoded["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings.cpu().numpy().astype(np.float32)

# --------------------------------------------------
# Build FAISS index for E5
# --------------------------------------------------
with open(ARTIFACT_DIR / "chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

chunk_texts = [c["text"] for c in chunks]

start_embed = time.perf_counter()
e5_vectors = embed_e5(chunk_texts, is_query=False)
embed_time = (time.perf_counter() - start_embed) * 1000

e5_dim = e5_vectors.shape[1]
assert e5_dim == 768, f"Expected 768 dimensions, got {e5_dim}"

e5_index = faiss.IndexFlatIP(e5_dim)
e5_index.add(e5_vectors)

faiss.write_index(e5_index, str(ARTIFACT_DIR / "e5_base.faiss"))

print(f"✅ Multilingual-E5 index built successfully!")
print(f"Dimension: {e5_dim}")
print(f"Indexed chunks: {e5_index.ntotal}")
print(f"Embedding latency for {len(chunks)} chunks: {embed_time:.2f} ms")

Loading intfloat/multilingual-e5-base onto cuda...


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Multilingual-E5 index built successfully!
Dimension: 768
Indexed chunks: 5
Embedding latency for 5 chunks: 97.70 ms


In [ ]:
import json
import unicodedata
from pathlib import Path

ARTIFACT_DIR = Path("/content/bge_m3_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 1. Authentic Telugu, English, and Code-Mixed News Articles
# ---------------------------------------------------------------------------
news_articles = [
    {
        "article_id": "art-001",
        "category": "Technology & Economy",
        "title": "Foxconn and Google expand investments in Hyderabad and Andhra Pradesh",
        "language": "en",
        "content": (
            "Taiwanese manufacturing giant Foxconn has announced a new phased investment "
            "expanding its electronics production footprint near Hyderabad. Concurrently, Google "
            "confirmed opening a state-of-the-art AI Operations Centre in Visakhapatnam to drive "
            "cloud services and skill over 20,000 local engineers in machine learning and data science."
        )
    },
    {
        "article_id": "art-002",
        "category": "Technology & Economy",
        "title": "హైదరాబాద్‌లో ఫాక్స్‌కాన్ భారీ పెట్టుబడుల విస్తరణ",
        "language": "te",
        "content": (
            "తైవాన్‌కు చెందిన ప్రముఖ ఎలక్ట్రానిక్స్ తయారీ సంస్థ ఫాక్స్‌కాన్ హైదరాబాద్ సమీపంలో తన నూతన ప్లాంట్ "
            "విస్తరణకు అదనపు నిధులు కేటాయించినట్లు ప్రకటించింది. ఈ ప్రాజెక్ట్ ద్వారా సుమారు 25,000 మంది స్థానిక "
            "యువతకు ఉపాధి అవకాశాలు లభిస్తాయని రాష్ట్ర ఐటీ శాఖ మంత్రి వెల్లడించారు. ఎలక్ట్రానిక్స్ మరియు మొబైల్ "
            "విడిభాగాల తయారీలో తెలంగాణను గ్లోబల్ హబ్‌గా తీర్చిదిద్దడమే తమ లక్ష్యమని పేర్కొన్నారు."
        )
    },
    {
        "article_id": "art-003",
        "category": "Entertainment & Tollywood",
        "title": "గ్లోబల్ బాక్సాఫీస్ వద్ద రామ్ చరణ్, ఎన్టీఆర్ నూతన చిత్రాల సందడి",
        "language": "te",
        "content": (
            "టాలీవుడ్ అగ్ర హీరోలు రామ్ చరణ్, జూనియర్ ఎన్టీఆర్ కొత్త చిత్రాలు అంతర్జాతీయ స్థాయిలో రికార్డు కలెక్షన్లు "
            "సాధిస్తున్నాయి. అమెరికా, బ్రిటన్, ఆస్ట్రేలియాలోని ప్రీమియర్ షోలకే ప్రేక్షకుల నుంచి భారీ స్పందన లభించింది. "
            "భారతీయ చిత్ర పరిశ్రమలో తెలుగు సినిమా మార్కెట్ సరికొత్త శిఖరాలకు చేరుకుందని సినీ విశ్లేషకులు అభిప్రాయపడుతున్నారు."
        )
    },
    {
        "article_id": "art-004",
        "category": "Entertainment & Tollywood",
        "title": "Tollywood Box Office Dominates Global Circuit with Multi-Lingual Releases",
        "language": "en",
        "content": (
            "Telugu cinema continues its winning streak across worldwide box offices, with major big-budget "
            "productions crossing milestone gross figures within their opening weekends. Overseas distribution "
            "rights have seen unprecedented bidding wars in North America and the Middle East, cementing South Indian "
            "filmmaking's pan-world appeal."
        )
    },
    {
        "article_id": "art-005",
        "category": "Agriculture & Climate",
        "title": "బంగాళాఖాతంలో అల్పపీడనం: కోస్తాంధ్ర, రాయలసీమలో భారీ వర్షాలు",
        "language": "te",
        "content": (
            "నైరుతి బంగాళాఖాతంలో ఏర్పడిన వాయుగుండం ప్రభావంతో ఆంధ్రప్రదేశ్‌లోని కోస్తాంధ్ర, రాయలసీమ జిల్లాల్లో "
            "రాగల 48 గంటల్లో భారీ నుంచి అతి భారీ వర్షాలు కురిసే అవకాశం ఉందని విశాఖ వాతావరణ కేంద్రం హెచ్చరించింది. "
            "తీరప్రాంత మత్స్యకారులు సముద్రంలోకి వేటకు వెళ్లవద్దని అధికారులు హెచ్చరికలు జారీ చేశారు."
        )
    },
    {
        "article_id": "art-006",
        "category": "Agriculture & Climate",
        "title": "Severe Low Pressure in Bay of Bengal: Heavy Rainfall Alert across Coastal AP",
        "language": "en",
        "content": (
            "The Meteorological Centre has sounded an orange alert for several coastal districts of Andhra Pradesh "
            "due to an intensifying low-pressure system in the Bay of Bengal. Farmers have been advised to take "
            "precautions for harvested kharif crops, while district emergency response teams have been put on high standby."
        )
    },
    {
        "article_id": "art-007",
        "category": "Infrastructure & Governance",
        "title": "అమరావతి రాజధాని పనుల పునరుద్ధరణపై కేబినెట్ కీలక నిర్ణయాలు",
        "language": "te",
        "content": (
            "ఆంధ్రప్రదేశ్ రాష్ట్ర రాజధాని అమరావతి నిర్మాణ పనులను వేగవంతం చేసేందుకు రాష్ట్ర మంత్రివర్గం కీలక ఆమోదం "
            "తెలిపింది. ప్రపంచ బ్యాంకు మరియు ఏడీబీ ఆర్థిక సహాయంతో ప్రధాన రహదారులు, ప్రభుత్వ సచివాలయం, అసెంబ్లీ "
            "భవనాల పనులను నిర్దేశిత గడువులోగా పూర్తి చేయాలని ప్రభుత్వం లక్ష్యంగా పెట్టుకుంది."
        )
    },
    {
        "article_id": "art-008",
        "category": "Infrastructure & Governance",
        "title": "Hyderabad Metro Phase 2 Detailed Project Report Finalized",
        "language": "en",
        "content": (
            "The Telangana state government has cleared the Detailed Project Report (DPR) for Hyderabad Metro Phase 2, "
            "stretching over 76 kilometers across five major corridors connecting Shamshabad Airport, Chandrayangutta, "
            "and the financial district. The project is anticipated to significantly reduce peak-hour traffic gridlock."
        )
    }
]

# ---------------------------------------------------------------------------
# 2. Chunking Function (Normalized with Metadata)
# ---------------------------------------------------------------------------
chunks = []
for art in news_articles:
    norm_content = unicodedata.normalize("NFC", art["content"]).strip()
    chunks.append({
        "chunk_id": f"chunk-{art['article_id']}",
        "article_id": art["article_id"],
        "category": art["category"],
        "language": art["language"],
        "title": art["title"],
        "text": f"{art['title']}\n{norm_content}"
    })

chunks_file = ARTIFACT_DIR / "real_news_corpus.jsonl"
with open(chunks_file, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

# ---------------------------------------------------------------------------
# 3. Ground-Truth Q&A Evaluation Pairs (Telugu, English, Code-Mixed)
# ---------------------------------------------------------------------------
eval_questions = [
    {
        "question_id": "eval-te-001",
        "language": "te",
        "question": "ఫాక్స్‌కాన్ విస్తరణ ద్వారా ఎంత మందికి ఉపాధి లభించనుంది?",
        "relevant_chunk_ids": ["chunk-art-002"]
    },
    {
        "question_id": "eval-en-001",
        "language": "en",
        "question": "Where is Google opening its new AI Operations Centre?",
        "relevant_chunk_ids": ["chunk-art-001"]
    },
    {
        "question_id": "eval-te-002",
        "language": "te",
        "question": "కోస్తాంధ్రలో భారీ వర్షాలకు కారణం ఏమిటి?",
        "relevant_chunk_ids": ["chunk-art-005"]
    },
    {
        "question_id": "eval-en-002",
        "language": "en",
        "question": "What is the planned length and route coverage of Hyderabad Metro Phase 2?",
        "relevant_chunk_ids": ["chunk-art-008"]
    },
    {
        "question_id": "eval-mixed-001",
        "language": "mixed",
        "question": "Amaravati capital construction కి World Bank funds ఇస్తున్నారా?",
        "relevant_chunk_ids": ["chunk-art-007"]
    },
    {
        "question_id": "eval-mixed-002",
        "language": "mixed",
        "question": "Tollywood movies overseas box office collections update ఏమిటి?",
        "relevant_chunk_ids": ["chunk-art-003", "chunk-art-004"]
    }
]

eval_file = ARTIFACT_DIR / "real_news_eval.jsonl"
with open(eval_file, "w", encoding="utf-8") as f:
    for q in eval_questions:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print(f"✅ Generated {len(chunks)} news chunks -> {chunks_file}")
print(f"✅ Generated {len(eval_questions)} test queries -> {eval_file}")

✅ Generated 8 news chunks -> /content/bge_m3_artifacts/real_news_corpus.jsonl
✅ Generated 6 test queries -> /content/bge_m3_artifacts/real_news_eval.jsonl


In [ ]:
import json
import time
import unicodedata
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

ARTIFACT_DIR = Path("/content/bge_m3_artifacts")
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------------------------------------------------------
# 1. Load Corpus and Benchmark Queries
# ---------------------------------------------------------------------------
with open(ARTIFACT_DIR / "real_news_corpus.jsonl", "r", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f if line.strip()]

with open(ARTIFACT_DIR / "real_news_eval.jsonl", "r", encoding="utf-8") as f:
    queries = [json.loads(line) for line in f if line.strip()]

corpus_texts = [c["text"] for c in corpus]
chunk_ids = [c["chunk_id"] for c in corpus]
id2idx = {cid: idx for idx, cid in enumerate(chunk_ids)}

print(f"Loaded {len(corpus)} chunks and {len(queries)} evaluation queries.")

# ---------------------------------------------------------------------------
# 2. Initialize Models (Self-contained)
# ---------------------------------------------------------------------------
# BGE-M3
if "bge_model" not in globals():
    from FlagEmbedding import BGEM3FlagModel
    bge_model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=torch.cuda.is_available())

# Multilingual-E5
E5_NAME = "intfloat/multilingual-e5-base"
if "e5_model" not in globals():
    e5_tokenizer = AutoTokenizer.from_pretrained(E5_NAME)
    e5_model = AutoModel.from_pretrained(E5_NAME).to(device)
    e5_model.eval()

def embed_bge(texts):
    norm_texts = [unicodedata.normalize("NFC", t).strip() for t in texts]
    res = bge_model.encode(norm_texts, max_length=512, return_dense=True, return_sparse=False, return_colbert_vecs=False)
    vecs = np.asarray(res["dense_vecs"], dtype=np.float32)
    faiss.normalize_L2(vecs)
    return vecs

def embed_e5(texts, is_query=False):
    prefix = "query: " if is_query else "passage: "
    prefixed = [prefix + unicodedata.normalize("NFC", t).strip() for t in texts]
    enc = e5_tokenizer(prefixed, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        out = e5_model(**enc)
        # Average pooling
        mask = enc["attention_mask"].unsqueeze(-1).expand(out[0].size()).float()
        vecs = torch.sum(out[0] * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        vecs = F.normalize(vecs, p=2, dim=1)
    return vecs.cpu().numpy().astype(np.float32)

# ---------------------------------------------------------------------------
# 3. Indexing Benchmark
# ---------------------------------------------------------------------------
print("\n--- Indexing Corpus ---")

# BGE-M3 Index
t0 = time.perf_counter()
bge_vectors = embed_bge(corpus_texts)
bge_index_time = (time.perf_counter() - t0) * 1000
bge_index = faiss.IndexFlatIP(1024)
bge_index.add(bge_vectors)

# E5 Index
t1 = time.perf_counter()
e5_vectors = embed_e5(corpus_texts, is_query=False)
e5_index_time = (time.perf_counter() - t1) * 1000
e5_index = faiss.IndexFlatIP(768)
e5_index.add(e5_vectors)

print(f"BGE-M3 (1024d) Indexing Time: {bge_index_time:.2f} ms")
print(f"E5-Base (768d)  Indexing Time: {e5_index_time:.2f} ms")

# ---------------------------------------------------------------------------
# 4. Evaluation Loop
# ---------------------------------------------------------------------------
def evaluate_model(model_name, embed_query_fn, index, top_k=3):
    per_query_results = []
    latencies = []
    lang_stats = defaultdict(lambda: {"recall1": [], "recall3": [], "rr": []})

    for q in queries:
        qid = q["question_id"]
        q_text = q["question"]
        targets = set(q["relevant_chunk_ids"])
        lang = q.get("language", "unknown")

        t_start = time.perf_counter()
        q_vec = embed_query_fn([q_text])
        scores, indices = index.search(q_vec, top_k)
        latency = (time.perf_counter() - t_start) * 1000
        latencies.append(latency)

        retrieved_ids = [chunk_ids[i] for i in indices[0] if i >= 0]

        # Metrics
        rec1 = 1.0 if retrieved_ids[:1] and retrieved_ids[0] in targets else 0.0
        rec3 = len(set(retrieved_ids[:3]) & targets) / len(targets) if targets else 0.0

        rr = 0.0
        for rank, cid in enumerate(retrieved_ids, start=1):
            if cid in targets:
                rr = 1.0 / rank
                break

        lang_stats[lang]["recall1"].append(rec1)
        lang_stats[lang]["recall3"].append(rec3)
        lang_stats[lang]["rr"].append(rr)

        per_query_results.append({
            "question_id": qid,
            "language": lang,
            "recall@1": rec1,
            "recall@3": rec3,
            "rr": rr,
            "latency_ms": latency
        })

    all_r1 = [r["recall@1"] for r in per_query_results]
    all_r3 = [r["recall@3"] for r in per_query_results]
    all_rr = [r["rr"] for r in per_query_results]

    summary = {
        "model": model_name,
        "overall": {
            "recall@1": float(np.mean(all_r1)),
            "recall@3": float(np.mean(all_r3)),
            "mrr": float(np.mean(all_rr)),
            "avg_latency_ms": float(np.mean(latencies))
        },
        "by_language": {
            l: {
                "recall@1": float(np.mean(stats["recall1"])),
                "recall@3": float(np.mean(stats["recall3"])),
                "mrr": float(np.mean(stats["rr"]))
            } for l, stats in lang_stats.items()
        },
        "details": per_query_results
    }
    return summary

bge_results = evaluate_model("BAAI/bge-m3", embed_bge, bge_index, top_k=3)
e5_results = evaluate_model("intfloat/multilingual-e5-base", lambda texts: embed_e5(texts, is_query=True), e5_index, top_k=3)

# ---------------------------------------------------------------------------
# 5. Display Research Comparison Table
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print(f"{'Metric':<25} {'BAAI/bge-m3 (1024d)':<22} {'Multilingual-E5 (768d)':<22}")
print("="*70)
print(f"{'Overall Recall@1':<25} {bge_results['overall']['recall@1']:<22.3f} {e5_results['overall']['recall@1']:<22.3f}")
print(f"{'Overall Recall@3':<25} {bge_results['overall']['recall@3']:<22.3f} {e5_results['overall']['recall@3']:<22.3f}")
print(f"{'Overall MRR':<25} {bge_results['overall']['mrr']:<22.3f} {e5_results['overall']['mrr']:<22.3f}")
print(f"{'Avg Latency (ms)':<25} {bge_results['overall']['avg_latency_ms']:<22.2f} {e5_results['overall']['avg_latency_ms']:<22.2f}")
print("-"*70)

for lang in ["te", "en", "mixed"]:
    b_mrr = bge_results["by_language"].get(lang, {}).get("mrr", 0.0)
    e_mrr = e5_results["by_language"].get(lang, {}).get("mrr", 0.0)
    print(f"{f'MRR [{lang.upper()}]':<25} {b_mrr:<22.3f} {e_mrr:<22.3f}")

print("="*70)

# Save final report to JSON
report_path = ARTIFACT_DIR / "bge_vs_e5_benchmark.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump({"bge_m3": bge_results, "multilingual_e5": e5_results}, f, indent=2, ensure_ascii=False)
print(f"\n✅ Benchmark report saved to: {report_path}")

Loaded 8 chunks and 6 evaluation queries.

--- Indexing Corpus ---
BGE-M3 (1024d) Indexing Time: 123.87 ms
E5-Base (768d)  Indexing Time: 114.11 ms

Metric                    BAAI/bge-m3 (1024d)    Multilingual-E5 (768d)
Overall Recall@1          1.000                  1.000                 
Overall Recall@3          1.000                  1.000                 
Overall MRR               1.000                  1.000                 
Avg Latency (ms)          57.18                  16.04                 
----------------------------------------------------------------------
MRR [TE]                  1.000                  1.000                 
MRR [EN]                  1.000                  1.000                 
MRR [MIXED]               1.000                  1.000                 

✅ Benchmark report saved to: /content/bge_m3_artifacts/bge_vs_e5_benchmark.json


In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

print("⏳ Loading local multilingual generator onto your Tesla T4 GPU...")
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ Local LLM loaded successfully into GPU VRAM!")

def local_voice_assistant(query_text, target_language="te"):
    # 1. BGE-M3 Retrieval
    t0 = time.perf_counter()
    q_vec = embed_bge([query_text])
    scores, pos = bge_index.search(q_vec, 2)
    retrieval_ms = (time.perf_counter() - t0) * 1000

    retrieved = [corpus[p] for p in pos[0] if p >= 0]
    context = "\n".join([f"- {c['title']}: {c['text']}" for c in retrieved])

    # 2. Local Prompt
    lang_req = "Answer in natural Telugu." if target_language == "te" else "Answer in natural English."
    messages = [
        {"role": "system", "content": f"You are the TeluguPeople voice news assistant. Give a concise, 1-2 sentence spoken answer based ONLY on the provided news context. Do not use bullet points or asterisks. {lang_req}"},
        {"role": "user", "content": f"News Context:\n{context}\n\nQuestion: {query_text}"}
    ]

    # 3. Generation
    t1 = time.perf_counter()
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=True,
            repetition_penalty=1.1
        )

    response = llm_tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()
    gen_ms = (time.perf_counter() - t1) * 1000

    return {
        "query": query_text,
        "source": retrieved[0]["title"],
        "retrieval_ms": round(retrieval_ms, 2),
        "gen_ms": round(gen_ms, 2),
        "response": response
    }

# ---------------------------------------------------------------------------
# Test Queries
# ---------------------------------------------------------------------------
print("\n" + "="*65)
r1 = local_voice_assistant("హైదరాబాద్‌లో ఫాక్స్‌కాన్ విస్తరణ వల్ల ఎంతమందికి ఉపాధి లభిస్తుంది?", target_language="te")
print(f"🗣️ Query:     {r1['query']}")
print(f"📰 Source:    {r1['source']} ({r1['retrieval_ms']} ms)")
print(f"⚡ LLM Gen:   {r1['gen_ms']} ms")
print(f"🎙️ Voice Out: {r1['response']}")

print("\n" + "="*65)
r2 = local_voice_assistant("Where is Google opening its new AI Operations Centre in Andhra Pradesh?", target_language="en")
print(f"🗣️ Query:     {r2['query']}")
print(f"📰 Source:    {r2['source']} ({r2['retrieval_ms']} ms)")
print(f"⚡ LLM Gen:   {r2['gen_ms']} ms")
print(f"🎙️ Voice Out: {r2['response']}")
print("="*65)

⏳ Loading local multilingual generator onto your Tesla T4 GPU...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Local LLM loaded successfully into GPU VRAM!

🗣️ Query:     హైదరాబాద్‌లో ఫాక్స్‌కాన్ విస్తరణ వల్ల ఎంతమందికి ఉపాధి లభిస్తుంది?
📰 Source:    హైదరాబాద్‌లో ఫాక్స్‌కాన్ భారీ పెట్టుబడుల విస్తరణ (109.32 ms)
⚡ LLM Gen:   4852.96 ms
🎙️ Voice Out: ఫాక్స్‌కాన్ హైదరాబాద్ సమీపంలో తన నూతన ప్లాంట్ విస్తరణకు అదనపు నిధుల�

🗣️ Query:     Where is Google opening its new AI Operations Centre in Andhra Pradesh?
📰 Source:    Foxconn and Google expand investments in Hyderabad and Andhra Pradesh (36.85 ms)
⚡ LLM Gen:   817.39 ms
🎙️ Voice Out: Google's new AI Operations Centre is being opened in Visakhapatnam, Andhra Pradesh.


In [ ]:
%pip install -q gTTS

from gtts import gTTS
import IPython.display as ipd

def speak_news(assistant_output, lang="te"):
    """
    Converts the assistant's text response to spoken audio and plays it in Colab.
    """
    text = assistant_output["response"]
    print(f"🎙️ Playing Audio [{lang.upper()}]: \"{text}\"")

    # Generate speech
    tts = gTTS(text=text, lang=lang, slow=False)
    audio_path = f"/content/bge_m3_artifacts/response_{lang}.mp3"
    tts.save(audio_path)

    # Render audio widget in Colab
    return ipd.Audio(audio_path, autoplay=True)

# ---------------------------------------------------------------------------
# Test Audio Playback
# ---------------------------------------------------------------------------
# 1. Ask a question
test_res = local_voice_assistant("హైదరాబాద్‌లో ఫాక్స్‌కాన్ విస్తరణ వల్ల ఎంతమందికి ఉపాధి లభిస్తుంది?", target_language="te")

# 2. Listen to the Telugu voice output!
speak_news(test_res, lang="te")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
🎙️ Playing Audio [TE]: "తౌర్પాటు మంత్రి వెల్లడించినప్పుడు ఫాక్స్‌కాన్ హైదరాబాద్ సమీపంలో తన"


In [ ]:
import shutil
from google.colab import files

# Create zip archive of all artifacts
archive_path = shutil.make_archive("/content/telugupeople_bge_m3_project", "zip", "/content/bge_m3_artifacts")
print(f"✅ Created archive: {archive_path}")

# Download directly to your computer
files.download(archive_path)

✅ Created archive: /content/telugupeople_bge_m3_project.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>